# Section 5 – Multivariate Analysis (Revised)

This notebook compares the five NIRF parameters: TLR, RPC, GO, OI and PR.

**Revisions in this version**
- Uses a reusable CSV loader and filter function instead of a fixed Colab-only path.
- Checks duplicate institution-year records before keeping one row.
- Question 4 uses a meaningful parameter-decline threshold (> 2 points by default), reports which parameters declined, and separates minor changes from material declines.
- Question 5 identifies and labels each institution's strongest and weakest standardized parameter.
- Uses Plotly for interactive charts.
- Z-scores are relative to the selected category/year sample; they are not absolute national performance scores.

## 1. Import Required Libraries

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import matplotlib.pyplot as plt

def load_master(path=None):
    """Load the NIRF master CSV from a supplied path or common Colab/local locations."""
    candidates = ([Path(path)] if path else []) + [
        Path("nirf_master_final.csv"),
        Path("/content/nirf_master_final.csv"),
        Path("/mnt/data/nirf_master_final.csv")
    ]
    for candidate in candidates:
        if candidate.exists():
            data = pd.read_csv(candidate)
            print(f"Loaded: {candidate} | shape={data.shape}")
            return data
    raise FileNotFoundError(
        "nirf_master_final.csv not found. Upload it to Colab or pass its full path to load_master(path)."
    )

def filter_data(data, **filters):
    """Reusable exact-value filters; e.g. filter_data(df, year=2025, category='Engineering')."""
    result = data.copy()
    for column, value in filters.items():
        if column not in result.columns:
            raise KeyError(f"Filter column '{column}' is not present.")
        result = result[result[column] == value]
    return result.copy()

df = load_master()

Loaded: /mnt/data/nirf_master_final.csv | shape=(11012, 42)


## 2. Load the NIRF Dataset

In [2]:
# Dataset is loaded by load_master() in the previous cell.
print("Rows:", len(df), "| Columns:", len(df.columns))
display(df.head())

Rows: 11012 | Columns: 42


,institution_key,institute_name,institute_id,institute_name_original,year,category,category_original,category_group,city,city_original,state,state_code,state_original,rank,rank_value,rank_status,rank_original,rank_low,rank_high,is_rank_band,rank_band_label,rank_band_source,rank_midpoint,score,TLR,RPC,GO,OI,PR,weight_scheme,is_top10,is_top50,is_top100,is_new_entry,first_year_in_category,category_first_year,categories_in_year,source_page,weight_scheme_basis,has_score,has_components,score_comparison_group
0,INST0936,Miranda House,IR17-COLL-1-29115,Miranda House,2017,College,College,General,Delhi,Delhi,Delhi,DL,Delhi,1.0,1.0,exact,1,1.0,1.0,False,NaN,NaN,1.0,69.39,75.76,27.44,90.06,96.45,41.82,30/20/25/15/10,True,True,True,False,2017,2017,6,CollegeRanking,estimated_from_data,True,True,College | 30/20/25/15/10
1,INST0814,Loyola College,IR17-COLL-2-959129,Loyola College,2017,College,College,General,Chennai,District Chennai,Tamil Nadu,TN,Tamil Nadu,2.0,2.0,exact,2,2.0,2.0,False,NaN,NaN,2.0,68.68,66.92,100.00,48.42,56.45,80.45,30/20/25/15/10,True,True,True,False,2017,2017,6,CollegeRanking,estimated_from_data,True,True,College | 30/20/25/15/10
2,INST1313,Shri Ram College of Commerce,IR17-COLL-2-29134,Shri Ram College of Commerce,2017,College,College,General,Delhi,Delhi,Delhi,DL,Delhi,3.0,3.0,exact,3,3.0,3.0,False,NaN,NaN,3.0,67.18,77.63,0.20,97.51,93.80,53.98,30/20/25/15/10,True,True,True,False,2017,2017,6,CollegeRanking,estimated_from_data,True,True,College | 30/20/25/15/10
3,INST0172,Bishop Heber College,IR17-COLL-1-19387,Bishop Heber College,2017,College,College,General,Tiruchirappalli,District Tiruchirapalli,Tamil Nadu,TN,Tamil Nadu,4.0,4.0,exact,4,4.0,4.0,False,NaN,NaN,4.0,61.18,76.62,53.99,61.28,68.11,18.63,30/20/25/15/10,True,True,True,False,2017,2017,6,CollegeRanking,estimated_from_data,True,True,College | 30/20/25/15/10
4,INST0093,Atma Ram Sanatan Dharma College,IR17-COLL-2-21799,Atma Ram Sanatan Dharma College,2017,College,College,General,New Delhi,New Delhi,Delhi,DL,Delhi,5.0,5.0,exact,5,5.0,5.0,False,NaN,NaN,5.0,60.68,74.22,30.54,79.60,82.51,0.21,30/20/25/15/10,True,True,True,False,2017,2017,6,CollegeRanking,estimated_from_data,True,True,College | 30/20/25/15/10


## 3. Understand the Dataset

In [3]:
print("Number of rows:", df.shape[0])
print("Number of columns:", df.shape[1])

print("\nColumn names:")
print(df.columns.tolist())

print("\nDataset information:")
df.info()

print("\nAvailable years:")
print(sorted(df["year"].dropna().unique()))

print("\nAvailable categories:")
print(sorted(df["category"].dropna().unique()))

print("\nRanking status:")
print(df["rank_status"].value_counts(dropna=False))

Number of rows: 11012
Number of columns: 42

Column names:
['institution_key', 'institute_name', 'institute_id', 'institute_name_original', 'year', 'category', 'category_original', 'category_group', 'city', 'city_original', 'state', 'state_code', 'state_original', 'rank', 'rank_value', 'rank_status', 'rank_original', 'rank_low', 'rank_high', 'is_rank_band', 'rank_band_label', 'rank_band_source', 'rank_midpoint', 'score', 'TLR', 'RPC', 'GO', 'OI', 'PR', 'weight_scheme', 'is_top10', 'is_top50', 'is_top100', 'is_new_entry', 'first_year_in_category', 'category_first_year', 'categories_in_year', 'source_page', 'weight_scheme_basis', 'has_score', 'has_components', 'score_comparison_group']

Dataset information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11012 entries, 0 to 11011
Data columns (total 42 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   institution_key          11012 non-null  object 
 1   institute

## 4. Select the Year and Category

In [4]:
selected_year = 2025
selected_category = "Engineering"

parameters = ["TLR", "RPC", "GO", "OI", "PR"]

print("Selected year:", selected_year)
print("Selected category:", selected_category)
print("Parameters:", parameters)

Selected year: 2025
Selected category: Engineering
Parameters: ['TLR', 'RPC', 'GO', 'OI', 'PR']


## 5. Prepare the Data for Analysis

Only institutions with exact published ranks are considered in the main ranking analysis.

Rank-band entries and entries without published exact ranks are excluded.

In [5]:
required_columns = ["institution_key","institute_name","year","category","rank",
                    "rank_status","score","score_comparison_group","weight_scheme"] + parameters
missing_columns = [c for c in required_columns if c not in df.columns]
if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

def prepare_category_year_data(data, year, category):
    result = filter_data(data, year=year, category=category)
    result = result[result["rank_status"].astype(str).str.strip().str.lower().eq("exact")].copy()
    for col in ["rank","score"] + parameters:
        result[col] = pd.to_numeric(result[col], errors="coerce")
    result = result.dropna(subset=["institution_key","institute_name","rank"])
    duplicate_keys = result[result.duplicated("institution_key", keep=False)]
    if not duplicate_keys.empty:
        print(f"Warning: {duplicate_keys['institution_key'].nunique()} duplicate institution keys found; retaining row with most non-missing analysis values.")
        result["_completeness"] = result[["score"] + parameters].notna().sum(axis=1)
        result = result.sort_values("_completeness").drop_duplicates("institution_key", keep="last").drop(columns="_completeness")
    return result

filtered_data = prepare_category_year_data(df, selected_year, selected_category)
print("Exact-ranked institutions:", len(filtered_data))
display(filtered_data.head())

Exact-ranked institutions: 100


,institution_key,institute_name,institute_id,institute_name_original,year,category,category_original,category_group,city,city_original,state,state_code,state_original,rank,rank_value,rank_status,rank_original,rank_low,rank_high,is_rank_band,rank_band_label,rank_band_source,rank_midpoint,score,TLR,RPC,GO,OI,PR,weight_scheme,is_top10,is_top50,is_top100,is_new_entry,first_year_in_category,category_first_year,categories_in_year,source_page,weight_scheme_basis,has_score,has_components,score_comparison_group
9955,INST0585,Indian Institute of Technology Madras,IR-E-U-0456,Indian Institute of Technology Madras,2025,Engineering,Engineering,Discipline,Chennai,Chennai,Tamil Nadu,TN,Tamil Nadu,1.0,1.0,exact,1,1.0,1.0,False,NaN,NaN,1.0,88.72,95.70,90.74,82.29,63.25,100.00,30/30/20/10/10,True,True,True,False,2017,2017,16,EngineeringRanking,estimated_from_data,True,True,Engineering | 30/30/20/10/10
9956,INST0574,Indian Institute of Technology Delhi,IR-E-I-1074,Indian Institute of Technology Delhi,2025,Engineering,Engineering,Discipline,New Delhi,New Delhi,Delhi,DL,Delhi,2.0,2.0,exact,2,2.0,2.0,False,NaN,NaN,2.0,85.74,85.89,93.47,81.67,62.01,93.97,30/30/20/10/10,True,True,True,False,2017,2017,16,EngineeringRanking,estimated_from_data,True,True,Engineering | 30/30/20/10/10
9957,INST0573,Indian Institute of Technology Bombay,IR-E-U-0306,Indian Institute of Technology Bombay,2025,Engineering,Engineering,Discipline,Mumbai,Mumbai,Maharashtra,MH,Maharashtra,3.0,3.0,exact,3,3.0,3.0,False,NaN,NaN,3.0,83.65,87.97,84.81,87.06,59.13,85.04,30/30/20/10/10,True,True,True,False,2017,2017,16,EngineeringRanking,estimated_from_data,True,True,Engineering | 30/30/20/10/10
9958,INST0583,Indian Institute of Technology Kanpur,IR-E-I-1075,Indian Institute of Technology Kanpur,2025,Engineering,Engineering,Discipline,Kanpur,Kanpur,Uttar Pradesh,UP,Uttar Pradesh,4.0,4.0,exact,4,4.0,4.0,False,NaN,NaN,4.0,81.82,91.77,76.34,88.08,59.18,78.49,30/30/20/10/10,True,True,True,False,2017,2017,16,EngineeringRanking,estimated_from_data,True,True,Engineering | 30/30/20/10/10
9959,INST0584,Indian Institute of Technology Kharagpur,IR-E-U-0573,Indian Institute of Technology Kharagpur,2025,Engineering,Engineering,Discipline,Kharagpur,Kharagpur,West Bengal,WB,West Bengal,5.0,5.0,exact,5,5.0,5.0,False,NaN,NaN,5.0,78.69,85.26,77.56,79.82,64.24,74.57,30/30/20/10/10,True,True,True,False,2017,2017,16,EngineeringRanking,estimated_from_data,True,True,Engineering | 30/30/20/10/10


## Question 1 – Comparison of the Top Five Institutions

Compare the top five institutions in the selected category based on their NIRF ranks.

Analyse their performance across TLR, RPC, GO, OI and PR.

In [6]:
def get_top_five(data):
    """
    Select the five highest-ranked institutions.
    A smaller rank number indicates better performance.
    """

    return (
        data.sort_values("rank", ascending=True)
        .head(5)
        .copy()
    )


top_five = get_top_five(filtered_data)

display(
    top_five[
        ["rank", "institute_name", "score"] + parameters
    ].reset_index(drop=True)
)

,rank,institute_name,score,TLR,RPC,GO,OI,PR
0,1.0,Indian Institute of Technology Madras,88.72,95.70,90.74,82.29,63.25,100.00
1,2.0,Indian Institute of Technology Delhi,85.74,85.89,93.47,81.67,62.01,93.97
2,3.0,Indian Institute of Technology Bombay,83.65,87.97,84.81,87.06,59.13,85.04
3,4.0,Indian Institute of Technology Kanpur,81.82,91.77,76.34,88.08,59.18,78.49
4,5.0,Indian Institute of Technology Kharagpur,78.69,85.26,77.56,79.82,64.24,74.57


### Visual Comparison of the Five NIRF Parameters

In [ ]:
def plot_top_five(data, year, category):
    if data.empty:
        print("No data available.")
        return
    plot_data = data.melt(
        id_vars=["institute_name"],
        value_vars=parameters,
        var_name="Parameter",
        value_name="Parameter Score"
    )
    fig = px.bar(
        plot_data, x="institute_name", y="Parameter Score",
        color="Parameter", barmode="group",
        hover_name="institute_name",
        hover_data={"Parameter": True, "Parameter Score":":.2f"},
        title=f"Top Five Institutions Across NIRF Parameters ({category}, {year})",
        labels={"institute_name":"Institution","Parameter Score":"Parameter score"}
    )
    fig.update_layout(xaxis_tickangle=-25, legend_title="NIRF parameter",
                      yaxis_range=[0,105], bargap=0.18)
    fig.show()

plot_top_five(top_five, selected_year, selected_category)


 ### Statistical Summary of the Top Five Institutions

In [8]:
parameter_summary = top_five[parameters].agg(
    ["mean", "median", "min", "max", "std"]
).round(2)

display(parameter_summary)

,TLR,RPC,GO,OI,PR
mean,89.32,84.58,83.78,61.56,86.41
median,87.97,84.81,82.29,62.01,85.04
min,85.26,76.34,79.82,59.13,74.57
max,95.70,93.47,88.08,64.24,100.00
std,4.38,7.65,3.59,2.34,10.57


## Question 2 – High Research Performance but Low Teaching Performance

Identify institutions with:

- RPC scores above the median RPC score.
- TLR scores below the median TLR score.

The medians are calculated using exact-ranked institutions with non-missing values for both RPC and TLR.

In [9]:
def analyse_high_research_low_teaching(data):

    analysis_data = data[
        ["rank", "institute_name", "score", "TLR", "RPC"]
    ].dropna(
        subset=["TLR", "RPC"]
    ).copy()

    if analysis_data.empty:
        return analysis_data, analysis_data.copy(), np.nan, np.nan

    research_median = analysis_data["RPC"].median()
    teaching_median = analysis_data["TLR"].median()

    selected = analysis_data[
        (analysis_data["RPC"] > research_median) &
        (analysis_data["TLR"] < teaching_median)
    ].copy()

    selected = selected.sort_values(
        "RPC",
        ascending=False
    )

    return (
        analysis_data,
        selected,
        research_median,
        teaching_median
    )


analysis_data_q2, high_research_low_teaching, research_median, teaching_median = (
    analyse_high_research_low_teaching(filtered_data)
)

print("Median RPC:", round(research_median, 2))
print("Median TLR:", round(teaching_median, 2))

print("\nInstitutions with high research but low teaching:")
display(high_research_low_teaching)

Median RPC: 42.8
Median TLR: 72.65

Institutions with high research but low teaching:


,rank,institute_name,score,TLR,RPC
9970,16.0,Vellore Institute of Technology,65.25,66.77,67.60
9991,37.0,Amity University - Gautam Budh Nagar,58.53,66.68,65.39
9974,20.0,Anna University,63.51,69.43,62.80
9964,10.0,Indian Institute of Technology (Banaras Hindu ...,67.24,66.39,61.68
9978,24.0,Jamia Millia Islamia,62.42,67.50,59.18
9997,43.0,University of Petroleum and Energy Studies,56.99,71.34,55.50
10006,52.0,Graphic Era University,55.26,68.63,55.33
10002,48.0,Lovely Professional University,55.99,63.04,52.88
10000,46.0,Symbiosis International,56.22,69.68,51.85
9996,42.0,Malaviya National Institute of Technology,57.45,69.29,51.84


### Research Performance vs Teaching Performance

In [ ]:
def plot_research_vs_teaching(
    analysis_data, selected, research_median, teaching_median, year, category
):
    if analysis_data.empty:
        print("No valid data available.")
        return
    plot_data = analysis_data.copy()
    plot_data["Group"] = np.where(
        plot_data.index.isin(selected.index),
        "High RPC and Low TLR", "Other Institutions"
    )
    fig = px.scatter(
        plot_data, x="TLR", y="RPC", color="Group",
        hover_name="institute_name",
        hover_data={"rank":True,"score":":.2f","TLR":":.2f","RPC":":.2f","Group":True},
        title=f"Research vs Teaching Performance ({category}, {year})",
        labels={"TLR":"Teaching, Learning and Resources (TLR)",
                "RPC":"Research and Professional Practice (RPC)"}
    )
    fig.add_vline(x=teaching_median, line_dash="dash", line_color="#D62728",
                  annotation_text=f"Median TLR: {teaching_median:.2f}")
    fig.add_hline(y=research_median, line_dash="dash", line_color="#2CA02C",
                  annotation_text=f"Median RPC: {research_median:.2f}")
    fig.update_layout(legend_title="Institution group")
    fig.show()

plot_research_vs_teaching(
    analysis_data_q2, high_research_low_teaching,
    research_median, teaching_median, selected_year, selected_category
)


### Percentage of Institutions with High Research but Low Teaching

In [11]:
total_q2 = len(analysis_data_q2)
identified_q2 = len(high_research_low_teaching)

percentage_q2 = (
    identified_q2 / total_q2 * 100
    if total_q2 > 0 else 0
)

print("Total institutions considered:", total_q2)
print("Institutions identified:", identified_q2)
print("Percentage:", round(percentage_q2, 2), "%")

print(
    "The percentage is calculated only among exact-ranked "
    "institutions with non-missing RPC and TLR scores."
)

Total institutions considered: 100
Institutions identified: 20
Percentage: 20.0 %
The percentage is calculated only among exact-ranked institutions with non-missing RPC and TLR scores.


## Question 3 – High Teaching Performance but Low Perception

Identify institutions with:

- TLR scores above the median TLR score.
- PR scores below the median PR score.

The medians are calculated using exact-ranked institutions with non-missing values for both TLR and PR.

In [12]:
def analyse_high_teaching_low_perception(data):

    analysis_data = data[
        ["rank", "institute_name", "score", "TLR", "PR"]
    ].dropna(
        subset=["TLR", "PR"]
    ).copy()

    if analysis_data.empty:
        return analysis_data, analysis_data.copy(), np.nan, np.nan

    teaching_median = analysis_data["TLR"].median()
    perception_median = analysis_data["PR"].median()

    selected = analysis_data[
        (analysis_data["TLR"] > teaching_median) &
        (analysis_data["PR"] < perception_median)
    ].copy()

    selected = selected.sort_values(
        "TLR",
        ascending=False
    )

    return (
        analysis_data,
        selected,
        teaching_median,
        perception_median
    )


analysis_data_q3, high_teaching_low_perception, teaching_median_q3, perception_median = (
    analyse_high_teaching_low_perception(filtered_data)
)

print("Median TLR:", round(teaching_median_q3, 2))
print("Median PR:", round(perception_median, 2))

print("\nInstitutions with high teaching but low perception:")
display(high_teaching_low_perception)

Median TLR: 72.65
Median PR: 17.8

Institutions with high teaching but low perception:


,rank,institute_name,score,TLR,PR
9981,27.0,Indian Institute of Technology Jodhpur,61.31,86.92,12.93
10010,56.0,Indian Institute of Technology Jammu,53.08,80.73,15.00
9994,40.0,Shanmugha Arts Science Technology & Research A...,58.02,80.47,10.76
9968,14.0,S.R.M. Institute of Science and Technology,65.83,78.69,17.45
9990,36.0,Kalinga Institute of Industrial Technology,58.79,78.64,10.76
10033,79.0,Sant Longowal Institute of Engineering & Techn...,48.42,78.11,0.72
9987,33.0,Kalasalingam Academy of Research and Education,59.03,78.03,7.27
10032,78.0,Rajiv Gandhi Institute of Petroleum Technology,48.52,77.78,0.72
10049,95.0,"C.V. Raman Global University, Odisha",46.62,77.56,11.86
9989,35.0,Koneru Lakshmaiah Education Foundation Univers...,58.95,77.55,4.12


### Teaching Performance vs Perception

In [ ]:
def plot_teaching_vs_perception(
    analysis_data, selected, teaching_median, perception_median, year, category
):
    if analysis_data.empty:
        print("No valid data available.")
        return
    plot_data = analysis_data.copy()
    plot_data["Group"] = np.where(
        plot_data.index.isin(selected.index),
        "High TLR and Low PR", "Other Institutions"
    )
    fig = px.scatter(
        plot_data, x="TLR", y="PR", color="Group",
        hover_name="institute_name",
        hover_data={"rank":True,"score":":.2f","TLR":":.2f","PR":":.2f","Group":True},
        title=f"Teaching vs Perception Performance ({category}, {year})",
        labels={"TLR":"Teaching, Learning and Resources (TLR)",
                "PR":"Perception (PR)"}
    )
    fig.add_vline(x=teaching_median, line_dash="dash", line_color="#D62728",
                  annotation_text=f"Median TLR: {teaching_median:.2f}")
    fig.add_hline(y=perception_median, line_dash="dash", line_color="#2CA02C",
                  annotation_text=f"Median PR: {perception_median:.2f}")
    fig.update_layout(legend_title="Institution group")
    fig.show()

plot_teaching_vs_perception(
    analysis_data_q3, high_teaching_low_perception,
    teaching_median_q3, perception_median, selected_year, selected_category
)


### Percentage of Institutions with High Teaching but Low Perception

In [14]:
total_q3 = len(analysis_data_q3)
identified_q3 = len(high_teaching_low_perception)

percentage_q3 = (
    identified_q3 / total_q3 * 100
    if total_q3 > 0 else 0
)

print("Total institutions considered:", total_q3)
print("Institutions identified:", identified_q3)
print("Percentage:", round(percentage_q3, 2), "%")

print(
    "The percentage is calculated only among exact-ranked "
    "institutions with non-missing TLR and PR scores."
)

Total institutions considered: 100
Institutions identified: 17
Percentage: 17.0 %
The percentage is calculated only among exact-ranked institutions with non-missing TLR and PR scores.


## Question 4 – Overall Score Improvement Despite a Decline in Parameters

Identify institutions whose overall NIRF score increased between two consecutive years, even though at least one individual parameter score declined.

Institutions are matched using institution_key.

To make the overall-score comparison meaningful, only records with the same score_comparison_group in both years are compared. This helps avoid comparing scores calculated under different parameter-weight schemes.

Changes in NIRF methodology or parameter weights can affect overall scores. Therefore, an increase in overall score does not necessarily mean that all individual parameters improved.

In [15]:
previous_year = 2024
current_year = 2025

if current_year != previous_year + 1:
    raise ValueError(
        "Select two consecutive years for comparison."
    )

print("Previous year:", previous_year)
print("Current year:", current_year)
print("Category:", selected_category)

Previous year: 2024
Current year: 2025
Category: Engineering


### Prepare Data for Year-to-Year Comparison

In [16]:
def prepare_year_comparison(
    data,
    category,
    previous_year,
    current_year
):
    """
    Match institutions across two consecutive years.

    Only exact-ranked institutions with the same
    score_comparison_group in both years are compared.
    """

    columns = [
        "institution_key",
        "institute_name",
        "year",
        "category",
        "rank",
        "rank_status",
        "score",
        "score_comparison_group",
        "weight_scheme"
    ] + parameters

    yearly_data = data[
        (data["category"] == category) &
        (data["year"].isin([previous_year, current_year]))
    ][columns].copy()

    yearly_data = yearly_data[
        yearly_data["rank_status"] == "exact"
    ].copy()

    numeric_columns = ["year", "rank", "score"] + parameters

    for column in numeric_columns:
        yearly_data[column] = pd.to_numeric(
            yearly_data[column],
            errors="coerce"
        )

    yearly_data = yearly_data.dropna(
        subset=[
            "institution_key",
            "score",
            "score_comparison_group"
        ]
    )

    yearly_data = yearly_data.drop_duplicates(
        subset=["institution_key", "year"],
        keep="last"
    )

    previous_data = yearly_data[
        yearly_data["year"] == previous_year
    ].copy()

    current_data = yearly_data[
        yearly_data["year"] == current_year
    ].copy()

    previous_data = previous_data.rename(
        columns={
            "institute_name": "previous_institute_name",
            "rank": "previous_rank",
            "score": "previous_score",
            "weight_scheme": "previous_weight_scheme",
            **{
                p: "previous_" + p
                for p in parameters
            }
        }
    )

    current_data = current_data.rename(
        columns={
            "institute_name": "current_institute_name",
            "rank": "current_rank",
            "score": "current_score",
            "weight_scheme": "current_weight_scheme",
            **{
                p: "current_" + p
                for p in parameters
            }
        }
    )

    # Match by institution identity and scoring comparison group
    comparison = current_data.merge(
        previous_data,
        on=[
            "institution_key",
            "score_comparison_group"
        ],
        how="inner",
        suffixes=("_current", "_previous")
    )

    return comparison


comparison = prepare_year_comparison(
    df,
    selected_category,
    previous_year,
    current_year
)

print(
    "Institutions with comparable scoring groups in both years:",
    len(comparison)
)

display(comparison.head())

Institutions with comparable scoring groups in both years: 93


,institution_key,current_institute_name,year_current,category_current,current_rank,rank_status_current,current_score,score_comparison_group,current_weight_scheme,current_TLR,current_RPC,current_GO,current_OI,current_PR,previous_institute_name,year_previous,category_previous,previous_rank,rank_status_previous,previous_score,previous_weight_scheme,previous_TLR,previous_RPC,previous_GO,previous_OI,previous_PR
0,INST0585,Indian Institute of Technology Madras,2025,Engineering,1.0,exact,88.72,Engineering | 30/30/20/10/10,30/30/20/10/10,95.70,90.74,82.29,63.25,100.00,Indian Institute of Technology Madras,2024,Engineering,1.0,exact,89.46,30/30/20/10/10,95.79,93.10,81.07,65.85,100.00
1,INST0574,Indian Institute of Technology Delhi,2025,Engineering,2.0,exact,85.74,Engineering | 30/30/20/10/10,30/30/20/10/10,85.89,93.47,81.67,62.01,93.97,Indian Institute of Technology Delhi,2024,Engineering,2.0,exact,86.66,30/30/20/10/10,85.71,92.20,84.52,65.39,98.53
2,INST0573,Indian Institute of Technology Bombay,2025,Engineering,3.0,exact,83.65,Engineering | 30/30/20/10/10,30/30/20/10/10,87.97,84.81,87.06,59.13,85.04,Indian Institute of Technology Bombay,2024,Engineering,3.0,exact,83.09,30/30/20/10/10,86.34,84.18,84.78,58.67,91.13
3,INST0583,Indian Institute of Technology Kanpur,2025,Engineering,4.0,exact,81.82,Engineering | 30/30/20/10/10,30/30/20/10/10,91.77,76.34,88.08,59.18,78.49,Indian Institute of Technology Kanpur,2024,Engineering,4.0,exact,82.79,30/30/20/10/10,94.25,74.08,89.05,60.62,84.19
4,INST0584,Indian Institute of Technology Kharagpur,2025,Engineering,5.0,exact,78.69,Engineering | 30/30/20/10/10,30/30/20/10/10,85.26,77.56,79.82,64.24,74.57,Indian Institute of Technology Kharagpur,2024,Engineering,5.0,exact,76.88,30/30/20/10/10,78.22,79.28,79.16,57.83,80.26


### Identify meaningful parameter declines

A tiny numerical decrease can flag an institution even when the change is negligible. This revised question uses a configurable default threshold of **more than 2 NIRF parameter points**. The output names the parameter(s) that declined. Change `DECLINE_THRESHOLD` if your project defines another meaningful threshold.

In [17]:
def find_improved_despite_decline(comparison, decline_threshold=2.0):
    result = comparison.copy()
    result["score_change"] = result["current_score"] - result["previous_score"]
    decline_columns = []
    for p in parameters:
        prev, curr = "previous_" + p, "current_" + p
        flag = p + "_material_decline"
        result[flag] = result[curr].notna() & result[prev].notna() & ((result[curr] - result[prev]) < -decline_threshold)
        decline_columns.append(flag)
    result["Declined Parameters"] = result.apply(
        lambda row: ", ".join([p for p in parameters if row[p + "_material_decline"]]), axis=1
    )
    result["Number of Material Declines"] = result[decline_columns].sum(axis=1)
    selected = result[(result["score_change"] > 0) & (result["Number of Material Declines"] > 0)].copy()
    return selected.sort_values(["score_change","Number of Material Declines"], ascending=[False,False])

DECLINE_THRESHOLD = 2.0
improved_despite_decline = find_improved_despite_decline(comparison, DECLINE_THRESHOLD)
print(f"Threshold: parameter decrease greater than {DECLINE_THRESHOLD} points.")
print("Institutions with overall score improvement and at least one material parameter decline:", len(improved_despite_decline))
display(improved_despite_decline[["current_institute_name","previous_score","current_score","score_change","Declined Parameters","Number of Material Declines"]].head(20))

Threshold: parameter decrease greater than 2.0 points.
Institutions with overall score improvement and at least one material parameter decline: 42


,current_institute_name,previous_score,current_score,score_change,Declined Parameters,Number of Material Declines
18,Indian Institute of Technology Patna,58.40,64.52,6.12,PR,1
38,Indian Institute of Technology Bhubaneswar,52.54,58.22,5.68,PR,1
37,International Institute of Information Technol...,54.29,58.45,4.16,PR,1
10,Birla Institute of Technology & Science -Pilani,63.04,67.02,3.98,PR,1
12,National Institute of Technology Rourkela,63.38,66.62,3.24,PR,1
85,Chitkara University - Rajpura,44.82,47.34,2.52,"OI, PR",2
69,Banasthali Vidyapith,47.86,50.38,2.52,GO,1
50,Graphic Era University,52.96,55.26,2.30,"GO, OI, PR",3
53,Dr. B. R. Ambedkar National Institute of Techn...,51.11,53.38,2.27,PR,1
90,"C.V. Raman Global University, Odisha",44.55,46.62,2.07,OI,1


### Display the Identified Institutions

In [18]:
result_columns = [
    "institution_key","previous_institute_name","current_institute_name",
    "previous_rank","current_rank","previous_score","current_score","score_change",
    "previous_weight_scheme","current_weight_scheme","Declined Parameters",
    "Number of Material Declines"
]
for parameter in parameters:
    result_columns.extend(["previous_"+parameter, "current_"+parameter,
                           parameter+"_material_decline"])
if not improved_despite_decline.empty:
    display(improved_despite_decline[result_columns].reset_index(drop=True))
else:
    print("No institutions meet the selected score-improvement and material-decline condition.")

,institution_key,previous_institute_name,current_institute_name,previous_rank,current_rank,previous_score,current_score,score_change,previous_weight_scheme,current_weight_scheme,Declined Parameters,Number of Material Declines,previous_TLR,current_TLR,TLR_material_decline,previous_RPC,current_RPC,RPC_material_decline,previous_GO,current_GO,GO_material_decline,previous_OI,current_OI,OI_material_decline,previous_PR,current_PR,PR_material_decline
0,INST0588,Indian Institute of Technology Patna,Indian Institute of Technology Patna,34.0,19.0,58.40,64.52,6.12,30/30/20/10/10,30/30/20/10/10,PR,1,67.22,79.24,False,49.41,55.04,False,77.73,78.40,False,53.00,63.95,False,25.69,21.73,True
1,INST0572,Indian Institute of Technology Bhubaneswar,Indian Institute of Technology Bhubaneswar,54.0,39.0,52.54,58.22,5.68,30/30/20/10/10,30/30/20/10/10,PR,1,58.61,74.57,False,35.88,39.14,False,74.96,75.07,False,58.35,62.69,False,33.72,28.27,True
2,INST0633,International Institute of Information Technol...,International Institute of Information Technol...,47.0,38.0,54.29,58.45,4.16,30/30/20/10/10,30/30/20/10/10,PR,1,65.47,75.84,False,40.89,43.22,False,66.90,70.72,False,63.14,63.32,False,26.90,22.58,True
3,INST0170,Birla Institute of Technology & Science -Pilani,Birla Institute of Technology & Science -Pilani,20.0,11.0,63.04,67.02,3.98,30/30/20/10/10,30/30/20/10/10,PR,1,73.19,82.07,False,52.49,56.41,False,79.47,81.04,False,58.14,60.41,False,36.28,32.29,True
4,INST1007,National Institute of Technology Rourkela,National Institute of Technology Rourkela,19.0,13.0,63.38,66.62,3.24,30/30/20/10/10,30/30/20/10/10,PR,1,76.49,83.55,False,57.00,59.25,False,73.24,76.89,False,59.41,61.74,False,27.38,22.37,True
5,INST0246,Chitkara University - Rajpura,Chitkara University - Rajpura,94.0,89.0,44.82,47.34,2.52,30/30/20/10/10,30/30/20/10/10,"OI, PR",2,52.61,50.97,False,36.19,49.46,False,51.54,51.93,False,64.91,62.79,True,13.82,5.41,True
6,INST0124,Banasthali Vidyapith,Banasthali Vidyapith,78.0,71.0,47.86,50.38,2.52,30/30/20/10/10,30/30/20/10/10,GO,1,64.78,70.41,False,21.00,24.74,False,73.66,71.47,True,72.16,71.39,False,1.78,4.12,False
7,INST0476,Graphic Era University,Graphic Era University,52.0,52.0,52.96,55.26,2.30,30/30/20/10/10,30/30/20/10/10,"GO, OI, PR",3,66.39,68.63,False,43.42,55.33,False,51.75,47.50,True,65.57,58.38,True,30.99,27.34,True
8,INST0324,Dr. B. R. Ambedkar National Institute of Techn...,Dr. B. R. Ambedkar National Institute of Techn...,58.0,55.0,51.11,53.38,2.27,30/30/20/10/10,30/30/20/10/10,PR,1,67.16,71.26,False,37.36,40.49,False,60.13,61.88,False,60.22,62.76,False,16.97,11.86,True
9,INST0182,"C.V. Raman Global University, Odisha","C.V. Raman Global University, Odisha",96.0,95.0,44.55,46.62,2.07,30/30/20/10/10,30/30/20/10/10,OI,1,76.20,77.56,False,4.85,10.31,False,57.48,59.54,False,74.22,71.59,True,13.17,11.86,False


### Visualize Overall Score Improvements

In [19]:
def plot_score_improvements(data, category):
    if data.empty:
        print("No institutions meet the selected threshold.")
        return
    plot_data = data.head(15).sort_values("score_change")
    fig = px.bar(plot_data, x="score_change", y="current_institute_name", orientation="h",
                 color="Number of Material Declines",
                 hover_data=["previous_score","current_score","Declined Parameters"],
                 title=f"Overall score improved despite material parameter declines – {category}",
                 labels={"score_change":"Overall score increase","current_institute_name":"Institution",
                         "Number of Material Declines":"No. of declining parameters"})
    fig.show()
plot_score_improvements(improved_despite_decline, selected_category)

## Question 5 – Identify the Strongest and Weakest Parameters

Identify institutions with the largest differences between their strongest and weakest NIRF parameters.

Raw parameter scores may have different distributions. Therefore, directly subtracting the minimum raw score from the maximum raw score can give a misleading comparison.

To improve comparability, each parameter is standardized within the selected category and year using a Z-score.

Standardized Performance Gap:

Highest parameter Z-score - Lowest parameter Z-score

A larger gap indicates a greater difference between an institution's relative strengths and weaknesses across the five parameters.

Only institutions with all five parameter values available are included.

### Calculate Standardized Parameter Scores

In [20]:
def calculate_standardized_gaps(data):

    gap_data = data[
        ["rank", "institute_name"] + parameters
    ].dropna(
        subset=parameters
    ).copy()

    if gap_data.empty:
        return gap_data

    standardized_columns = []

    for parameter in parameters:

        mean_value = gap_data[parameter].mean()
        std_value = gap_data[parameter].std(ddof=0)

        z_column = parameter + "_Z"

        if std_value == 0 or pd.isna(std_value):

            gap_data[z_column] = 0.0

        else:

            gap_data[z_column] = (
                gap_data[parameter] - mean_value
            ) / std_value

        standardized_columns.append(z_column)

    gap_data["Strongest Parameter"] = (
        gap_data[standardized_columns]
        .idxmax(axis=1)
        .str.replace("_Z", "", regex=False)
    )

    gap_data["Weakest Parameter"] = (
        gap_data[standardized_columns]
        .idxmin(axis=1)
        .str.replace("_Z", "", regex=False)
    )

    gap_data["Strongest Z-score"] = (
        gap_data[standardized_columns].max(axis=1)
    )

    gap_data["Weakest Z-score"] = (
        gap_data[standardized_columns].min(axis=1)
    )

    gap_data["Standardized Performance Gap"] = (
        gap_data["Strongest Z-score"] -
        gap_data["Weakest Z-score"]
    )

    return gap_data


gap_data = calculate_standardized_gaps(filtered_data)

print(
    "Institutions with all five parameter scores:",
    len(gap_data)
)

display(gap_data.head())

Institutions with all five parameter scores: 100


,rank,institute_name,TLR,RPC,GO,OI,PR,TLR_Z,RPC_Z,GO_Z,OI_Z,PR_Z,Strongest Parameter,Weakest Parameter,Strongest Z-score,Weakest Z-score,Standardized Performance Gap
9955,1.0,Indian Institute of Technology Madras,95.70,90.74,82.29,63.25,100.00,2.643098,2.764801,1.389788,0.316850,3.762801,PR,OI,3.762801,0.316850,3.445950
9956,2.0,Indian Institute of Technology Delhi,85.89,93.47,81.67,62.01,93.97,1.536321,2.926686,1.331358,0.150327,3.467954,PR,OI,3.467954,0.150327,3.317627
9957,3.0,Indian Institute of Technology Bombay,87.97,84.81,87.06,59.13,85.04,1.770989,2.413160,1.839323,-0.236436,3.031307,PR,OI,3.031307,-0.236436,3.267743
9958,4.0,Indian Institute of Technology Kanpur,91.77,76.34,88.08,59.18,78.49,2.199710,1.910900,1.935450,-0.229721,2.711034,PR,OI,2.711034,-0.229721,2.940755
9959,5.0,Indian Institute of Technology Kharagpur,85.26,77.56,79.82,64.24,74.57,1.465244,1.983244,1.157010,0.449800,2.519359,PR,OI,2.519359,0.449800,2.069559


### Find Institutions with the Largest Standardized Gaps

In [21]:
largest_gaps = gap_data.sort_values(
    "Standardized Performance Gap",
    ascending=False
).head(10).copy()

display(
    largest_gaps[
        [
            "rank",
            "institute_name",
            "Strongest Parameter",
            "Weakest Parameter",
            "Strongest Z-score",
            "Weakest Z-score",
            "Standardized Performance Gap"
        ]
    ].reset_index(drop=True)
)

,rank,institute_name,Strongest Parameter,Weakest Parameter,Strongest Z-score,Weakest Z-score,Standardized Performance Gap
0,88.0,AU College of Engineering,OI,RPC,2.333927,-1.836184,4.170111
1,45.0,Saveetha Institute of Medical and Technical Sc...,OI,GO,1.387163,-2.541050,3.928213
2,1.0,Indian Institute of Technology Madras,PR,OI,3.762801,0.316850,3.445950
3,95.0,"C.V. Raman Global University, Odisha",OI,RPC,1.436852,-2.004592,3.441443
4,76.0,Christ University,OI,GO,0.287306,-3.112157,3.399463
5,34.0,Aligarh Muslim University,RPC,OI,0.684010,-2.645647,3.329658
6,2.0,Indian Institute of Technology Delhi,PR,OI,3.467954,0.150327,3.317627
7,3.0,Indian Institute of Technology Bombay,PR,OI,3.031307,-0.236436,3.267743
8,20.0,Anna University,PR,OI,1.842630,-1.312121,3.154751
9,93.0,Panjab University,OI,TLR,0.346395,-2.786990,3.133385


### Visualize the Largest Standardized Performance Gaps

In [22]:
def plot_standardized_gaps(data, year, category):
    if data.empty:
        print("No institutions have all five parameter values.")
        return
    top = data.nlargest(15, "Standardized Performance Gap").copy()
    top["Weakest → Strongest"] = top["Weakest Parameter"] + " → " + top["Strongest Parameter"]
    fig = px.bar(top.sort_values("Standardized Performance Gap"), x="Standardized Performance Gap",
                 y="institute_name", orientation="h", color="Weakest Parameter",
                 hover_data=["Weakest → Strongest","Strongest Z-score","Weakest Z-score"],
                 title=f"Institutions with largest standardized parameter gaps ({category}, {year})",
                 labels={"institute_name":"Institution","Standardized Performance Gap":"Z-score gap"})
    fig.show()
plot_standardized_gaps(gap_data, selected_year, selected_category)

## 6. Overall Summary

This section analysed the multivariate performance of institutions using five NIRF parameters.

The analysis covered five questions:

1. Comparison of the top five institutions across TLR, RPC, GO, OI and PR.
2. Identification of institutions with high research performance but low teaching performance.
3. Identification of institutions with high teaching performance but low perception.
4. Identification of institutions whose overall scores improved despite declines in one or more parameters.
5. Identification of institutions with the largest standardized differences between their strongest and weakest parameters.

Exact-ranked institutions were used for the main analysis.

For Questions 2 and 3, medians and percentages were calculated only using institutions with non-missing values for the two relevant parameters.

For Question 4, institutions were matched using institution_key, and overall scores were compared only when the score_comparison_group was the same in both years.

For Question 5, Z-scores were used instead of raw parameter-score differences. This provides a standardized measure of relative strengths and weaknesses within the selected category and year.

The functions developed in this section can be reused in an interactive dashboard by passing different years and categories.

In [23]:
print("="*60)
print("SECTION 5 – MULTIVARIATE ANALYSIS (REVISED)")
print("="*60)
print("Selected category/year:", selected_category, selected_year)
print("Q1 top-five institutions:", len(top_five))
print("Q2 high RPC / low TLR:", len(high_research_low_teaching))
print("Q3 high TLR / low PR:", len(high_teaching_low_perception))
print("Q4 material decline threshold:", DECLINE_THRESHOLD)
print("Q4 qualifying institutions:", len(improved_despite_decline))
print("Q5 institutions with all five parameters:", len(gap_data))
print("Interpretation: standardized gaps are relative within the selected sample.")

SECTION 5 – MULTIVARIATE ANALYSIS (REVISED)
Selected category/year: Engineering 2025
Q1 top-five institutions: 5
Q2 high RPC / low TLR: 20
Q3 high TLR / low PR: 17
Q4 material decline threshold: 2.0
Q4 qualifying institutions: 42
Q5 institutions with all five parameters: 100
Interpretation: standardized gaps are relative within the selected sample.
